# 01 — Gaussian mixture forward model

This notebook confirms that the production forward model, `tools.data.gaussians.GaussianMixture`, reconstructs a known multi-Gaussian elevation profile exactly from its parameters, and that its three evaluation entry points agree with one another.

Modules exercised:

- `tools.data.gaussians.GaussianMixture` (`evaluate_pixel`, `evaluate_slice`, `evaluate_batch`)

All inputs are synthetic, the random seed is fixed, and the computation is CPU only.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

plt.rcParams.update({
    "figure.dpi"     : 110,
    "savefig.dpi"    : 110,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.30,
    "grid.linewidth" : 0.4,
})

print("Repo root :", REPO_ROOT)
print("NumPy     :", np.__version__)


In [ ]:
def gaussian_mixture_profile(height_axis, components, noise_std=0.0, rng=None):
    profile = np.zeros_like(height_axis, dtype=np.float64)
    for amp, mu, sigma in components:
        profile += amp * np.exp(-((height_axis - mu) ** 2) / (2.0 * sigma * sigma))
    if noise_std > 0.0:
        draw    = rng.normal(0.0, noise_std, size=height_axis.shape) if rng is not None else np.random.normal(0.0, noise_std, size=height_axis.shape)
        profile = profile + draw
    return profile.astype(np.float32)


def pack_parameters(components_per_pixel, k_max, shape):
    Az, R  = shape
    params = np.zeros((3 * k_max, Az, R), dtype=np.float32)
    for (az, rg), comps in components_per_pixel.items():
        for k, (amp, mu, sigma) in enumerate(comps[:k_max]):
            params[3 * k    , az, rg] = amp
            params[3 * k + 1, az, rg] = mu
            params[3 * k + 2, az, rg] = sigma
    return params


## Real forward model import

`GaussianMixture` packs a pixel parameter vector as a flat stack of `(amplitude, mu, sigma)` triples, i.e. index `3*k` is amplitude, `3*k+1` is the centroid height, `3*k+2` is the spread of component `k`.

In [ ]:
from tools.data.gaussians import GaussianMixture

K           = 3
height_axis = np.linspace(0.0, 40.0, 200).astype(np.float64)

true_components = [
    (1.00, 10.0, 2.0),
    (0.60, 22.0, 3.5),
    (0.35, 31.0, 1.5),
]

params = np.zeros(3 * K, dtype=np.float64)
for k, (amp, mu, sigma) in enumerate(true_components):
    params[3 * k    ] = amp
    params[3 * k + 1] = mu
    params[3 * k + 2] = sigma

print('parameter vector:', params)

## Per-pixel reconstruction and components

`evaluate_pixel` returns the summed profile and the individual component curves. We overlay them on the analytic ground-truth profile built by the notebook helper. The summed model should sit exactly on the ground truth.

In [ ]:
total, comps = GaussianMixture.evaluate_pixel(params, height_axis, K)
truth        = gaussian_mixture_profile(height_axis, true_components)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(height_axis, truth, color='black', lw=2.0, label='ground truth')
ax.plot(height_axis, total, color='tab:red', lw=1.4, ls='--', label='evaluate_pixel sum')
for k, comp in enumerate(comps):
    ax.plot(height_axis, comp, lw=0.9, alpha=0.8, label=f'component {k + 1}')
ax.set_xlabel('height h [m]')
ax.set_ylabel('backscatter intensity')
ax.set_title('Forward model reconstruction vs ground truth')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

max_abs_err = float(np.max(np.abs(total - truth)))
print('max absolute error (pixel sum vs truth):', max_abs_err)

## Cross-check the three evaluators

`evaluate_slice` evaluates the mixture at one height across a spatial map, and `evaluate_batch` evaluates a batch of pixels over the whole axis. Sampling `evaluate_slice` at every height of a single-pixel map must reproduce the `evaluate_pixel` curve, and `evaluate_batch` must match it too.

In [ ]:
params_map = params.astype(np.float32).reshape(3 * K, 1, 1)
slice_curve = np.array([float(GaussianMixture.evaluate_slice(params_map, float(h), K)[0, 0]) for h in height_axis])

amps = np.array([[c[0] for c in true_components]], dtype=np.float32)
mus  = np.array([[c[1] for c in true_components]], dtype=np.float32)
sigs = np.array([[c[2] for c in true_components]], dtype=np.float32)
batch_curve = GaussianMixture.evaluate_batch(height_axis.astype(np.float32), amps, mus, sigs)[0]

print('slice vs pixel  max abs diff:', float(np.max(np.abs(slice_curve - total))))
print('batch vs pixel  max abs diff:', float(np.max(np.abs(batch_curve - total))))

## Expected outcome

The dashed `evaluate_pixel` sum should lie on top of the black ground-truth curve with a maximum absolute error at the level of floating-point rounding. The slice and batch evaluators should agree with the per-pixel evaluator to the same precision, confirming all three forward-model entry points are numerically consistent.